# NULL Handling

## Overview
NULL represents the **absence of a value** — not zero, not empty string, not false. NULL propagates through arithmetic, comparisons, string functions, and most aggregates.

**Core NULL functions:**

| Function | Behaviour |
|---|---|
| `IS NULL` / `IS NOT NULL` | Only correct way to test for NULL |
| `COALESCE(a, b, c, ...)` | Returns first non-NULL argument |
| `NULLIF(a, b)` | Returns NULL if a = b, otherwise a |
| `IFNULL(a, b)` | SQLite shorthand for `COALESCE(a, b)` |

**NULL propagation rules:**
- `NULL = NULL` → NULL (use `IS NULL`)
- `NULL <> 'x'` → NULL (not TRUE)
- `NULL + 5` → NULL
- `'hello' || NULL` → NULL
- `COALESCE(NULL, NULL, 3)` → 3

**PostgreSQL:** `IS DISTINCT FROM` / `IS NOT DISTINCT FROM` treat NULLs as equal — useful for nullable foreign key comparisons.

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE intake_raw (
    record_id INTEGER PRIMARY KEY, patient_ref TEXT, first_name TEXT, last_name TEXT,
    dob TEXT, sex TEXT, province TEXT, phone TEXT, email TEXT,
    cost_str TEXT, intake_date TEXT, notes TEXT
);
CREATE TABLE lab_raw (
    lab_id INTEGER PRIMARY KEY, patient_ref TEXT, test_code TEXT,
    result_txt TEXT, collected TEXT, status TEXT
);
CREATE TABLE provider_raw (
    prov_id INTEGER PRIMARY KEY, name_raw TEXT, dept_code TEXT,
    start_dt TEXT, salary TEXT
);
INSERT INTO intake_raw VALUES
  (1,'P-001','aroha','Ngata','1985-03-12','F','NB','506-555-0101','aroha@mail.com','$3,200.00','2023-01-15','Referred by GP'),
  (2,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (3,'P-003','FATIMA','AL-RASHID','1990-07-22','female','Ontario','416-555-0303',NULL,'120.00','2023-03-05','Annual checkup'),
  (4,'P-004','James','MacLeod','Jan 30 1955','M','NB',NULL,'james@mail.com','$5,500','18-03-2023','Surgery follow-up'),
  (5,'P-005','sofia','Petrov','2001-09-15','F','BC','604-555-0505','sofia@mail.com','$95.00','2023-04-02',NULL),
  (6,'P-006','Noah','Williams','08/06/1968','m','AB ','780-555-0606','noah@mail.com','780','11/05/2023',NULL),
  (7,'P-007','Mei','Zhang','1995-04-17','F','ON','416-555-0707',NULL,'$2,100.00','22-06-2023','Follow-up required'),
  (8,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (9,'P-008','ethan','tremblay','01/12/1980',NULL,'QC','514-555-0808','ethan@mail.com','80.00','14-07-2023',NULL),
  (10,'P-009','Priya','Nair','1977-08-25','F','bc',NULL,'priya@mail.com','$1,750.00','01/10/2023',NULL),
  (11,'P-010','Marcus','Okafor','1993-05-19','M','ON','647-555-1010',NULL,'2200','03-11-2023',NULL),
  (12,'P-011','Diana','Flores','14/02/2000','female','NB','506-555-1111','diana@mail.com',NULL,'2023-12-01',NULL),
  (13,NULL,'Unknown',NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,'Incomplete record'),
  (14,'','Test','Record','2023-01-01','X','XX','000-000-0000','test@test.com','-1','2023-01-01','Test entry');
INSERT INTO lab_raw VALUES
  (1,'P-001','HBA1C','7.2 %','2023-03-10','Active'),
  (2,'P-002','HBA1C','9.1%','2023-03-15','active'),
  (3,'P-003','CREAT','88.5umol/L','2023-04-01','ACTIVE'),
  (4,'P-004','CREAT','145 umol/L','2023-04-12','Active'),
  (5,'P-005','HBA1C','5.8 %','2023-05-05',''),
  (6,'P-006','SODIUM','138mmol/L','2023-05-20','Active'),
  (7,'P-007','SODIUM','151 mmol/L','2023-06-01',NULL),
  (8,'P-001','CREAT','72.3 umol/L','2023-06-18','Active'),
  (9,'P-008','HBA1C','6.4%','2023-07-02','Active'),
  (10,'P-009','CREAT','210umol/L','2023-07-15','active');
INSERT INTO provider_raw VALUES
  (1,'MacNeil, Sarah MD','CARD','2018-01-15','$95,000'),
  (2,'Dr. James Wong','ONCO','2019-03-01','88000'),
  (3,'Fatima Osei M.D.','FAM','2017-06-01','$82,500.00'),
  (4,'Larson, Ethan','ORTH','2020-09-15','91000.00'),
  (5,'Sharma, Priya MD','EMRG','2016-11-01','$78,000'),
  (6,'Lucas Petit','RAD','2021-02-28',NULL);
""")
conn.commit()

print("intake_raw row count:", conn.execute("SELECT COUNT(*) FROM intake_raw").fetchone()[0])
print()
print("=== NULL/blank counts per column ===")
cols = ["patient_ref","first_name","last_name","dob","sex","province","phone","email","cost_str","intake_date"]
for c in cols:
    nulls = conn.execute(f"SELECT COUNT(*) FROM intake_raw WHERE {c} IS NULL OR {c} = ''").fetchone()[0]
    print(f"  {c}: {nulls} null/blank")

intake_raw row count: 14

=== NULL/blank counts per column ===
  patient_ref: 2 null/blank
  first_name: 0 null/blank
  last_name: 1 null/blank
  dob: 1 null/blank
  sex: 2 null/blank
  province: 1 null/blank
  phone: 3 null/blank
  email: 4 null/blank
  cost_str: 2 null/blank
  intake_date: 1 null/blank


---
## IS NULL / IS NOT NULL — the only correct test

In [2]:
# Demonstrate that = NULL never works
print("=== NULL comparison traps ===")
q_wrong = "SELECT COUNT(*) FROM intake_raw WHERE sex = NULL"
q_right = "SELECT COUNT(*) FROM intake_raw WHERE sex IS NULL"
print(f"WHERE sex = NULL  (wrong):   {conn.execute(q_wrong).fetchone()[0]} rows")
print(f"WHERE sex IS NULL (correct): {conn.execute(q_right).fetchone()[0]} rows")
print()
print("= NULL always evaluates to NULL (unknown), never TRUE.")

# Find records missing critical fields
print()
print("=== Records with at least one critical field missing ===")
q = """
SELECT record_id, patient_ref, first_name, last_name, dob, sex
FROM   intake_raw
WHERE  patient_ref IS NULL OR patient_ref = ''
    OR first_name  IS NULL
    OR last_name   IS NULL
    OR dob         IS NULL
    OR sex         IS NULL
"""
print(pd.read_sql(q, conn).to_string(index=False))

=== NULL comparison traps ===
WHERE sex = NULL  (wrong):   0 rows
WHERE sex IS NULL (correct): 2 rows

= NULL always evaluates to NULL (unknown), never TRUE.

=== Records with at least one critical field missing ===
 record_id patient_ref first_name last_name        dob  sex
         9       P-008      ethan  tremblay 01/12/1980 None
        13        None    Unknown      None       None None
        14                   Test    Record 2023-01-01    X


---
## COALESCE — first non-NULL value

In [3]:
print("=== COALESCE to substitute defaults ===")
q = """
SELECT  record_id,
        COALESCE(patient_ref, '(no ref)')           AS patient_ref,
        COALESCE(sex, 'Unknown')                     AS sex_clean,
        COALESCE(NULLIF(province,''), 'Unknown')   AS province_clean,
        COALESCE(phone, email, '(no contact)')       AS best_contact,
        COALESCE(cost_str, '0')                      AS cost_safe
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print("=== COALESCE prevents NULL propagation in arithmetic ===")
q2 = """
SELECT  record_id,
        cost_str,
        COALESCE(
            CAST(REPLACE(REPLACE(cost_str,'$',''),',','') AS REAL),
            0.0
        ) AS cost_numeric
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q2, conn).to_string(index=False))

=== COALESCE to substitute defaults ===
 record_id patient_ref sex_clean province_clean   best_contact cost_safe
         1       P-001         F             NB   506-555-0101 $3,200.00
         2       P-002      Male             NS   902-555-0202      1850
         3       P-003    female        Ontario   416-555-0303    120.00
         4       P-004         M             NB james@mail.com    $5,500
         5       P-005         F             BC   604-555-0505    $95.00
         6       P-006         m            AB    780-555-0606       780
         7       P-007         F             ON   416-555-0707 $2,100.00
         8       P-002      Male             NS   902-555-0202      1850
         9       P-008   Unknown             QC   514-555-0808     80.00
        10       P-009         F             bc priya@mail.com $1,750.00
        11       P-010         M             ON   647-555-1010      2200
        12       P-011    female             NB   506-555-1111         0
        13 

---
## NULLIF — converting sentinel values to NULL

In [4]:
print("=== NULLIF: turn blanks and sentinel values into NULL ===")
q = """
SELECT  record_id,
        patient_ref,
        NULLIF(patient_ref, '')               AS ref_nulled_if_blank,
        NULLIF(TRIM(province), 'XX')          AS province_nulled_if_invalid,
        COALESCE(NULLIF(TRIM(province),''), 'Unknown') AS province_clean
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print("=== NULLIF prevents divide-by-zero ===")
q2 = """
SELECT  record_id, cost_str,
        CAST(REPLACE(REPLACE(COALESCE(cost_str,'0'),'$',''),',','') AS REAL) AS cost_num,
        100.0 / NULLIF(
            CAST(REPLACE(REPLACE(COALESCE(cost_str,'0'),'$',''),',','') AS REAL), 0
        ) AS safe_inverse
FROM    intake_raw WHERE record_id <= 6
"""
print(pd.read_sql(q2, conn).to_string(index=False))
conn.close()

=== NULLIF: turn blanks and sentinel values into NULL ===
 record_id patient_ref ref_nulled_if_blank province_nulled_if_invalid province_clean
         1       P-001               P-001                         NB             NB
         2       P-002               P-002                         NS             NS
         3       P-003               P-003                    Ontario        Ontario
         4       P-004               P-004                         NB             NB
         5       P-005               P-005                         BC             BC
         6       P-006               P-006                         AB             AB
         7       P-007               P-007                         ON             ON
         8       P-002               P-002                         NS             NS
         9       P-008               P-008                         QC             QC
        10       P-009               P-009                         bc             bc
       

---
## Common Pitfalls

**1. Testing for NULL with = or <> returns zero rows**
`WHERE sex = NULL` always returns nothing. Every comparison with NULL evaluates to NULL (unknown). Use `WHERE sex IS NULL` without exception.

**2. NOT IN with a NULL in the subquery returns zero rows**
`WHERE province NOT IN ('ON', NULL)` returns nothing because `province <> NULL` is NULL for every row. Add `WHERE col IS NOT NULL` inside the subquery, or switch to NOT EXISTS.

**3. Empty string is not NULL**
`''` passes an `IS NOT NULL` check. Many applications insert `''` instead of NULL. Normalise with `NULLIF(TRIM(col), '')` before any IS NULL checks or COALESCE calls.

**4. COALESCE short-circuits — order matters for performance**
Arguments are evaluated left to right and stop at the first non-NULL. Put the cheapest, most-likely-non-NULL expression first.

**5. String concatenation with NULL produces NULL**
`first_name || ' ' || last_name` is NULL if either name is NULL. Use `COALESCE(first_name,'') || ' ' || COALESCE(last_name,'')` for safe concatenation.

**6. AVG silently excludes NULLs — denominator shrinks**
`AVG(cost_numeric)` over 10 rows where 3 are NULL averages over 7. If NULLs mean zero, use `AVG(COALESCE(cost_numeric, 0))` and document the assumption.


---
*sql_methods_library - Samantha McGarrigle*